# Calibrate PaySim Parameters

This notebook fits the source generator parameters that are genuinely available from PaySim: transaction type ratios, log-normal amount parameters per type, and fraud rates. Channel, device, location, and currency behavior remain explicit assumptions in `src/generate/config.py` because PaySim does not contain those fields.

Expected dataset: Kaggle `ealaxi/paysim1`. If Kaggle credentials are unavailable, download the CSV manually and place it under `data/external/paysim/`.

In [1]:
from __future__ import annotations

import json
import os
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external' / 'paysim'
WORKSPACE_KAGGLE_DIR = PROJECT_ROOT / 'data' / 'external' / 'kaggle'
PARAMS_PATH = PROJECT_ROOT / 'src' / 'generate' / 'calibration' / 'paysim_params.json'
EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)
WORKSPACE_KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
PARAMS_PATH.parent.mkdir(parents=True, exist_ok=True)

home_kaggle_dir = Path.home() / '.kaggle'
if (home_kaggle_dir / 'kaggle.json').exists():
    os.environ['KAGGLE_CONFIG_DIR'] = str(home_kaggle_dir)
else:
    os.environ['KAGGLE_CONFIG_DIR'] = str(WORKSPACE_KAGGLE_DIR)
    print(f'Using workspace Kaggle config dir: {WORKSPACE_KAGGLE_DIR}')
    print('Place kaggle.json there if the default home .kaggle path is unavailable.')

DATASET = 'ealaxi/paysim1'
TYPE_MAP = {
    'TRANSFER': 'transfer',
    'CASH_OUT': 'withdrawal',
    'PAYMENT': 'payment',
    'CASH_IN': 'topup',
    'DEBIT': 'debit',
}
PROJECT_TYPES = ['payment', 'transfer', 'withdrawal', 'topup', 'debit']

# PaySim amounts are in synthetic units, not VND. The distribution SHAPE (sigma) and the relative
# scale across transaction types are realistic, but the absolute magnitude is far too small read as
# VND directly (payments would be a few thousand VND, and the per-type cap would clip everything).
# AMOUNT_SCALE multiplies every PaySim amount before fitting so the resulting mu / sigma / cap land
# in a realistic Vietnamese fintech range while preserving the shape. Tune this one number to make
# all amounts bigger or smaller. With SCALE=20: payment median ~158k VND, transfer ~8.6M, ATM
# withdrawal ~2.3M.
AMOUNT_SCALE = 20.0

Using workspace Kaggle config dir: C:\Users\vquoc\Desktop\data lakehouse for fintech\data\external\kaggle
Place kaggle.json there if the default home .kaggle path is unavailable.


In [2]:
def find_paysim_csv() -> Path | None:
    csv_files = sorted(EXTERNAL_DIR.glob('*.csv'))
    return csv_files[0] if csv_files else None


csv_path = find_paysim_csv()
if csv_path is None:
    print('PaySim CSV not found locally. Trying Kaggle download...')
    subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', DATASET, '-p', str(EXTERNAL_DIR), '--unzip'],
        check=True,
    )
    csv_path = find_paysim_csv()

if csv_path is None:
    raise FileNotFoundError(f'No PaySim CSV found in {EXTERNAL_DIR}. Download {DATASET} manually and rerun.')

csv_path

WindowsPath('C:/Users/vquoc/Desktop/data lakehouse for fintech/data/external/paysim/PS_20174392719_1491204439457_log.csv')

In [3]:
usecols = ['type', 'amount', 'isFraud']
df = pd.read_csv(csv_path, usecols=usecols)
df['project_type'] = df['type'].map(TYPE_MAP)
df = df.dropna(subset=['project_type'])
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
df = df[df['amount'].notna() & (df['amount'] >= 0)]
df.shape, df.head()

((6362620, 4),
        type    amount  isFraud project_type
 0   PAYMENT   9839.64        0      payment
 1   PAYMENT   1864.28        0      payment
 2  TRANSFER    181.00        1     transfer
 3  CASH_OUT    181.00        1   withdrawal
 4   PAYMENT  11668.14        0      payment)

In [4]:
type_ratios = (
    df['project_type']
    .value_counts(normalize=True)
    .reindex(PROJECT_TYPES, fill_value=0.0)
    .to_dict()
)

amount_lognormal = {}
fraud_amount_lognormal = {}
for txn_type, group in df.groupby('project_type'):
    # Scale PaySim synthetic units into a realistic VND magnitude before fitting (see AMOUNT_SCALE).
    amount_vnd = group['amount'].to_numpy(dtype='float64') * AMOUNT_SCALE
    logged = np.log(amount_vnd + 1.0)
    amount_lognormal[txn_type] = {
        'mu': round(float(logged.mean()), 6),
        'sigma': round(float(logged.std(ddof=1)), 6),
        'cap_vnd': int(max(1.0, np.quantile(amount_vnd, 0.999))),
    }

    fraud_mask = group['isFraud'].to_numpy() == 1
    if fraud_mask.any():
        fraud_logged = np.log(amount_vnd[fraud_mask] + 1.0)
        fraud_amount_lognormal[txn_type] = {
            'mu': round(float(fraud_logged.mean()), 6),
            'sigma': round(float(fraud_logged.std(ddof=1)), 6),
        }

fraud_rate = float(df['isFraud'].mean())
fraud_rate_by_type = (
    df.groupby('project_type')['isFraud']
    .mean()
    .reindex(PROJECT_TYPES, fill_value=0.0)
    .to_dict()
)

params = {
    'source': 'paysim_calibrated',
    'source_dataset': DATASET,
    'source_file': str(csv_path.relative_to(PROJECT_ROOT)),
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'amount_scale_factor': AMOUNT_SCALE,
    'notes': [
        'PaySim is used only for transaction type ratios, amount log-normal parameters, and fraud rates.',
        'PaySim amounts are multiplied by amount_scale_factor before fitting to reach a realistic VND magnitude while keeping the distribution shape.',
        'The production generator lifts fraud_rate to 0.7 percent through config for a denser portfolio demo.',
    ],
    'pay_sim_observed_fraud_rate': round(fraud_rate, 8),
    'demo_fraud_rate': 0.007,
    'type_ratios': {k: round(float(v), 8) for k, v in type_ratios.items()},
    'fraud_rate_by_type': {k: round(float(v), 8) for k, v in fraud_rate_by_type.items()},
    'amount_lognormal': amount_lognormal,
    'fraud_amount_lognormal': fraud_amount_lognormal,
}

with PARAMS_PATH.open('w', encoding='utf-8') as handle:
    json.dump(params, handle, ensure_ascii=False, indent=2)

PARAMS_PATH, params

(WindowsPath('C:/Users/vquoc/Desktop/data lakehouse for fintech/src/generate/calibration/paysim_params.json'),
 {'source': 'paysim_calibrated',
  'source_dataset': 'ealaxi/paysim1',
  'source_file': 'data\\external\\paysim\\PS_20174392719_1491204439457_log.csv',
  'generated_at': '2026-06-10T14:44:38.800684+00:00',
  'amount_scale_factor': 20.0,
  'notes': ['PaySim is used only for transaction type ratios, amount log-normal parameters, and fraud rates.',
   'PaySim amounts are multiplied by amount_scale_factor before fitting to reach a realistic VND magnitude while keeping the distribution shape.',
   'The production generator lifts fraud_rate to 0.7 percent through config for a denser portfolio demo.'],
  'pay_sim_observed_fraud_rate': 0.00129082,
  'demo_fraud_rate': 0.007,
  'type_ratios': {'payment': 0.33814608,
   'transfer': 0.08375622,
   'withdrawal': 0.35166331,
   'topup': 0.21992261,
   'debit': 0.00651178},
  'fraud_rate_by_type': {'payment': 0.0,
   'transfer': 0.00768799,